# WP30 — Causal World Model
## Layer 11: Neural MLP Transition Learning for Meta-Level Planning

---

This notebook demonstrates **WP30: Causal World Model**, which replaces the
tabular per-action mean in WP20's trajectory model with a compact MLP that learns
state-dependent transition dynamics.

### The Gap WP30 Closes

WP20's `SynthesisTrajectoryModel` predicts accuracy deltas as tabular per-action
sample means. This is **state-blind**: the predicted delta for DEMOTE_WORST is the
same regardless of whether min(probs) is 0.35 (safe) or 0.05 (near-collapse).

### WP30 Solution: Neural World Model

$$\text{MLP}: (\phi(s_t), a_{\text{one-hot}}) \rightarrow \Delta s$$

| Component | Role |
|-----------|------|
| `ReplayBuffer` | Fixed-capacity ring buffer of `TrajectoryTransition` |
| `LatentTransitionModel` | 2-layer MLP (pure Python, no PyTorch) trained online |
| `WorldModelRollout` | K-step lookahead using neural model (or tabular fallback) |
| `WorldModelCRLS` | InvariantCRLS + WP30 neural world model layer |

### Theoretical Grounding
> *Ha & Schmidhuber (2018) World Models: a compact latent-space transition model
> allows 'dreaming' (planning in latent space) before acting in the real environment.*

Runtime: **~8 min** (no GPU required)

In [ ]:
# ── 0. Environment setup ────────────────────────────────────────────────────
import sys, os, importlib

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        os.system('git clone https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
    print(f'Local mode — repo root: {repo_root}')

import warnings; warnings.filterwarnings('ignore')
import time, random, json, math
import numpy as np
import matplotlib.pyplot as plt

from prometheus.wp30_causal_world_model import (
    WorldModelCRLS, LatentTransitionModel, ReplayBuffer,
    WorldModelRecord, WorldModelRollout,
    verify_wp30_exit_criteria,
)
from prometheus.wp22_bandit_exploration import BanditMode
from prometheus.wp17_crls_synthesis import SynthesisAction
from prometheus.environments.go import GoBoard

import prometheus
print(f'Prometheus version: {prometheus.__version__}')
print('WP30 imports OK.')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 7), 'font.size': 11})
SEED = 42; random.seed(SEED); np.random.seed(SEED)

In [ ]:
# ── 1. Configuration ────────────────────────────────────────────────────────
QUICK_MODE      = True
N_GENERATIONS   = 12 if QUICK_MODE else 30
PUZZLES_PER_GEN = 25 if QUICK_MODE else 60
BOARD_SIZE      = 9

REGIME_SEQUENCE = (
    ['ATARI'] * 4 + ['TERRITORY'] * 4 + ['MIXED'] * (N_GENERATIONS - 8)
)[:N_GENERATIONS]

print(f'Mode: {"QUICK" if QUICK_MODE else "FULL"}')
print(f'Generations: {N_GENERATIONS} | Regimes: {REGIME_SEQUENCE}')

In [ ]:
# ── 2. Puzzle factory ────────────────────────────────────────────────────────
def make_atari_puzzle(board_size, rng):
    board = GoBoard(size=board_size)
    cx = board_size // 2
    stones = [(cx, cx), (cx, cx+1), (cx+1, cx)]
    for r, c in stones:
        if board.is_on_board(r, c) and board.board[r, c] == GoBoard.EMPTY:
            board.board[r, c] = GoBoard.BLACK
    all_libs = set()
    for r, c in stones:
        for nr, nc in board.get_neighbors(r, c):
            if board.board[nr, nc] == GoBoard.EMPTY:
                all_libs.add((nr, nc))
    libs = list(all_libs); rng.shuffle(libs)
    for r, c in libs[:-1]: board.board[r, c] = GoBoard.WHITE
    board.current_player = GoBoard.WHITE
    return board, GoBoard.WHITE, libs[-1]

def make_territory_puzzle(board_size, rng):
    board = GoBoard(size=board_size)
    for r, c in [(0, 0), (0, board_size-1), (board_size-1, 0)]:
        board.board[r, c] = GoBoard.WHITE
    board.current_player = GoBoard.BLACK
    return board, GoBoard.BLACK, (board_size-1, board_size-1)

PUZZLE_FACTORIES = {'ATARI': make_atari_puzzle, 'TERRITORY': make_territory_puzzle}

def generate_puzzles(regime, n, board_size, seed):
    rng = np.random.default_rng(seed)
    if regime == 'MIXED':
        return [(PUZZLE_FACTORIES['ATARI'] if i % 2 == 0 else PUZZLE_FACTORIES['TERRITORY'])(board_size, rng) for i in range(n)]
    return [PUZZLE_FACTORIES[regime](board_size, rng) for _ in range(n)]

print('Puzzle factory OK.')

---
## Section 1 — WorldModelCRLS (11 Layers)

The `WorldModelCRLS` extends `InvariantCRLS` (WP27) with:
1. A `ReplayBuffer` that accumulates `TrajectoryTransition` records
2. A `LatentTransitionModel` (2-layer MLP) trained online via mini-batch SGD
3. A `WorldModelRollout` that uses the MLP for planning when the buffer is large enough

The 11-tuple from `end_of_generation()` includes a `WorldModelRecord` tracking
prediction MAE and whether the neural or tabular model was used.

In [ ]:
# ── 3. Instantiate WorldModelCRLS ─────────────────────────────────────────────
STRATEGIES = ['ATARI', 'LADDER', 'TERRITORY', 'MIXED']

stack = WorldModelCRLS(
    strategies     = STRATEGIES,
    bandit_mode    = BanditMode.UCB1,
    replay_capacity = 200,
    min_buffer      = 5,     # start using neural model after 5 transitions
    mlp_hidden      = 16,    # hidden layer width
    mlp_lr          = 0.01,
    mlp_batch_size  = 8,
)

print('WorldModelCRLS (11 layers) instantiated.')
print('  Layers: WP17→WP19→WP20→WP21→WP22→WP23→WP24→WP25→WP26→WP27→WP30')
print(f'  Replay buffer capacity: 200')
print(f'  MLP hidden size: 16 | LR: 0.01 | Batch: 8')
print(f'  Neural model active after: 5 buffer entries')

accuracies, wm_records, prediction_maes, rollout_sources = [], [], [], []

In [ ]:
# ── 4. Main experiment loop ─────────────────────────────────────────────────
print('=' * 75)
print(f'  Gen  Regime        Acc     Buffer  MAE     RolloutSource')
print('=' * 75)

for gen in range(N_GENERATIONS):
    regime = REGIME_SEQUENCE[gen]
    puzzles = generate_puzzles(regime, PUZZLES_PER_GEN, BOARD_SIZE, seed=gen * 137 + 1)

    acc = stack.run_generation(puzzles)
    eleven_tuple = stack.end_of_generation()
    wm_rec = eleven_tuple[-1]   # WorldModelRecord is the last element

    accuracies.append(acc)
    wm_records.append(wm_rec)

    buf_size     = stack.replay_buffer.size
    mae          = wm_rec.prediction_mae if wm_rec else float('nan')
    rollout_src  = wm_rec.rollout_source  if wm_rec else 'fallback'
    prediction_maes.append(mae)
    rollout_sources.append(rollout_src)

    print(
        f'  {gen:3d}  {regime:<12}  {acc:.3f}  '
        f'{buf_size:5d}   '
        f'{mae:.4f}  '
        f'{rollout_src}'
    )

print('=' * 75)
print(f'Mean accuracy: {np.mean(accuracies):.3f}')
print(f'Final buffer size: {stack.replay_buffer.size}')
neural_gens = sum(1 for s in rollout_sources if s == 'neural')
print(f'Neural rollout used: {neural_gens}/{N_GENERATIONS} generations')

In [ ]:
# ── 5. Visualisation ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
gens = list(range(N_GENERATIONS))

regime_colors = {'ATARI': '#fff3cd', 'TERRITORY': '#d1ecf1', 'MIXED': '#e2d9f3'}
def add_bands(ax):
    prev, start = None, 0
    for g, r in enumerate(REGIME_SEQUENCE):
        if r != prev:
            if prev: ax.axvspan(start-.5, g-.5, alpha=.2, color=regime_colors.get(prev,'#eee'))
            start, prev = g, r
    ax.axvspan(start-.5, N_GENERATIONS-.5, alpha=.2, color=regime_colors.get(prev,'#eee'))

# A: Accuracy
ax = axes[0, 0]
add_bands(ax)
ax.plot(gens, accuracies, 'g-o', linewidth=2.5, markersize=6)
ax.axhline(np.mean(accuracies), color='darkgreen', linestyle='--', alpha=0.7,
           label=f'Mean={np.mean(accuracies):.3f}')
ax.set_ylabel('Accuracy'); ax.set_title('Accuracy under Regime Shifts', fontweight='bold')
ax.legend(); ax.set_ylim(0, 1.05)

# B: MLP prediction MAE over time
ax2 = axes[0, 1]
add_bands(ax2)
maes_clean = [m if not (isinstance(m, float) and math.isnan(m)) else None for m in prediction_maes]
valid_gens  = [g for g, m in zip(gens, maes_clean) if m is not None]
valid_maes  = [m for m in maes_clean if m is not None]
if valid_maes:
    ax2.plot(valid_gens, valid_maes, 'b-o', linewidth=2.5, markersize=6,
             label='MLP prediction MAE')
    ax2.fill_between(valid_gens, 0, valid_maes, alpha=0.2, color='blue')
ax2.set_ylabel('Mean Absolute Error (Δacc prediction)')
ax2.set_title('Neural World Model Prediction Error\n(decreasing = MLP is learning dynamics)', fontweight='bold')
ax2.legend()

# C: Replay buffer size over time
buf_sizes = []
# Reconstruct from wm_records
for i, rec in enumerate(wm_records):
    buf_sizes.append(min(i + 1, 200))   # approximate: 1 transition/gen, capped at capacity
ax3 = axes[1, 0]
add_bands(ax3)
ax3.plot(gens, buf_sizes, 'purple', linewidth=2.5, marker='D', markersize=5,
         label='Buffer size')
ax3.axhline(5, color='orange', linestyle='--', label='min_buffer (neural activates)')
ax3.set_xlabel('Generation'); ax3.set_ylabel('Replay buffer size')
ax3.set_title('Replay Buffer Growth\n(orange = threshold for neural rollout)', fontweight='bold')
ax3.legend()

# D: Rollout source distribution
from collections import Counter
ax4 = axes[1, 1]
ax4.axis('off')
src_counts = Counter(rollout_sources)
summary_text = (
    'WP30 Causal World Model — Summary\n'
    '═══════════════════════════════════\n\n'
    f'  Generations:       {N_GENERATIONS}\n'
    f'  Final buffer size: {stack.replay_buffer.size}\n'
    f'  Neural rollouts:   {src_counts.get("neural", 0)}\n'
    f'  Tabular fallbacks: {src_counts.get("tabular", 0)}\n'
    f'  Hard fallbacks:    {src_counts.get("fallback", 0)}\n\n'
    f'  Mean accuracy:     {np.mean(accuracies):.3f}\n'
    f'  Final MAE:         {valid_maes[-1]:.4f}\n\n' if valid_maes else '\n'
    'MLP architecture:\n'
    '  Input:  φ(s) + a_one_hot\n'
    '  Hidden: 16 units (ReLU)\n'
    '  Output: Δs (acc + probs)\n\n'
    'Ha & Schmidhuber (2018):\n'
    '  World Models — dream before\n'
    '  acting; plan in latent space.\n\n'
    'Sutton & Barto (2018) Dyna-Q:\n'
    '  Internal model = synthetic\n'
    '  experience for planning.'
)
ax4.text(0.05, 0.95, summary_text, transform=ax4.transAxes,
         fontsize=9.5, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

fig.suptitle(
    'WP30: Causal World Model — Prometheus v0\n'
    'Layer 11: Neural MLP transition learning for state-dependent meta-planning',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('wp30_causal_world_model.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to wp30_causal_world_model.png')

In [ ]:
# ── 6. Verify WP30 exit criteria ─────────────────────────────────────────────
results = verify_wp30_exit_criteria(stack)
print('WP30 Exit Criteria Verification')
print('=' * 50)
all_pass = True
for criterion, passed in results.items():
    status = '✓ PASS' if passed else '✗ FAIL'
    print(f'  {status}  {criterion}')
    if not passed: all_pass = False
print()
if all_pass:
    print('All WP30 exit criteria satisfied.')
    print('The neural world model is learning state-dependent transition dynamics.')
else:
    print('Some criteria not yet met — run more generations.')

---
## Conclusions

**Neural world models** (Ha & Schmidhuber 2018, Sutton & Barto Dyna-Q) allow the
synthesis stack to *plan in latent space* before committing to an action. The MLP
transition model learns the state-dependent effect of each synthesis action —
overcoming the state-blindness of WP20's tabular approach.

This is a realisation of Good's vision: an ultraintelligent machine that *models*
its own improvement dynamics can plan its self-modifications rather than merely
reacting to them.

### References
- Ha, D. & Schmidhuber, J. (2018). World models. *arXiv:1803.10122*.
- Sutton, R.S. & Barto, A.G. (2018). *Reinforcement Learning: An Introduction*, 2nd ed. MIT Press. §8 Dyna-Q.
- Rumelhart, D.E., Hinton, G.E. & Williams, R.J. (1986). Learning representations by back-propagation. *Nature*, 323, 533–536.
- Good, I.J. (1965). Speculations concerning the first ultraintelligent machine.